In [ ]:
import pandas as pd

FILEPATH = '/content/ai_model_predictions.csv'
OUTPUT = '/content/aipd_sample.csv'
TARGET_ROWS = 150000
CHUNK_SIZE = 200000

# Columns we actually need
KEEP_COLS = [
    'doc_id', 'flag_patent', 'pub_dt',
    'predict86_any_ai',
    'ai_score_ml', 'ai_score_evo', 'ai_score_nlp',
    'ai_score_speech', 'ai_score_vision', 'ai_score_planning',
    'ai_score_kr', 'ai_score_hardware'
]

chunks = []
total = 0

for chunk in pd.read_csv(FILEPATH, chunksize=CHUNK_SIZE, usecols=KEEP_COLS):
    # Filter: high-confidence AI patents, post-1990
    chunk['pub_dt'] = pd.to_datetime(chunk['pub_dt'], errors='coerce')
    filtered = chunk[
        (chunk['predict86_any_ai'] == 1) &
        (chunk['pub_dt'].dt.year >= 1990)
    ]
    chunks.append(filtered)
    total += len(filtered)
    print(f"  Collected {total:,} rows so far...")
    if total >= TARGET_ROWS:
        break

df = pd.concat(chunks).head(TARGET_ROWS)
df['year'] = df['pub_dt'].dt.year

# Derive dominant AI category per patent
score_cols = {
    'Machine Learning': 'ai_score_ml',
    'Computer Vision': 'ai_score_vision',
    'NLP': 'ai_score_nlp',
    'Planning & Control': 'ai_score_planning',
    'Knowledge Rep.': 'ai_score_kr',
    'Speech Recognition': 'ai_score_speech',
    'AI Hardware': 'ai_score_hardware',
    'Evolutionary Comp.': 'ai_score_evo'
}
df['dominant_category'] = df[[v for v in score_cols.values()]].idxmax(axis=1)
df['dominant_category'] = df['dominant_category'].map({v: k for k, v in score_cols.items()})

df.to_csv(OUTPUT, index=False)
print(f"\n✅ Done! Saved {len(df):,} rows to {OUTPUT}")
print(df['dominant_category'].value_counts())
print(df['year'].describe())

  Collected 36,517 rows so far...
  Collected 75,737 rows so far...
  Collected 115,673 rows so far...
  Collected 156,995 rows so far...

✅ Done! Saved 150,000 rows to /content/aipd_sample.csv
dominant_category
Planning & Control    42911
Computer Vision       37672
AI Hardware           29052
NLP                   14758
Knowledge Rep.        12026
Speech Recognition     5357
Machine Learning       4944
Evolutionary Comp.     3280
Name: count, dtype: int64
count    150000.000000
mean       2019.141073
std           0.721453
min        2018.000000
25%        2019.000000
50%        2019.000000
75%        2020.000000
max        2020.000000
Name: year, dtype: float64
